In [ ]:
#@title JAVA INSTALLATION { run: "auto" }

#@markdown
#@markdown 설치할 자바 버전을 선택하세요. (1.21+ 버전은 Java 21 권장)
java_version = "21" #@param ["8", "11", "16", "17", "21"]

import os

# 서버 폴더 생성 및 이동
!mkdir -p /content/server
%cd /content/server

print(f"Installing OpenJDK {java_version}...")

# 선택한 버전에 따라 패키지 설치
if java_version == "8":
    !sudo apt-get update -y && sudo apt-get install openjdk-8-jre-headless -y
elif java_version == "11":
    !sudo apt-get update -y && sudo apt-get install openjdk-11-jre-headless -y
else:
    # 16, 17, 21 버전 처리
    pkg_name = f"openjdk-{java_version}-jre-headless"
    !sudo apt-get update -y && sudo apt-get install {pkg_name} -y

print(f"\n--- Java {java_version} 설치 완료 ---")
!java -version

In [ ]:
#@title DOWNLOAD SERVER FILE & EULA { run: "auto" }

#@markdown
#@markdown 다운로드할 서버 파일(.jar)의 링크를 입력하세요.
#@markdown (입력하지 않으면 Paper 1.21.1 최신 빌드가 다운로드됩니다.)
download_url = "https://api.papermc.io/v2/projects/paper/versions/1.21.1/builds/121/downloads/paper-1.21.1-121.jar" #@param {type:"string"}

import os

# 서버 폴더 이동 확인
%cd /content/server

# 파일 다운로드 (이름을 paper.jar로 고정하여 이후 스크립트 호환성 유지)
print("Downloading server file...")
!wget -O paper.jar "{download_url}"

# EULA 동의 파일 생성
with open('eula.txt', 'w') as f:
    f.write('eula=true')

    print("\n--- Download Complete & EULA Accepted ---")
    # 다운로드된 파일 확인
    !ls -lh paper.jar

In [ ]:
#@title SERVER.PROPERTIES { run: "auto" }

#@markdown ### [ 기본 설정 ]
gamemode = "survival" #@param ["survival", "creative", "adventure", "spectator"]
difficulty = "normal" #@param ["peaceful", "easy", "normal", "hard"]
max_players = 20 #@param {type:"integer"}
motd = "A Minecraft Server" #@param {type:"string"}
server_port = 25565 #@param {type:"integer"}
online_mode = True #@param {type:"boolean"}
white_list = False #@param {type:"boolean"}

#@markdown ### [ 고급 설정 ]
pvp = True #@param {type:"boolean"}
allow_flight = False #@param {type:"boolean"}
level_seed = "" #@param {type:"string"}
view_distance = 10 #@param {type:"slider", min:4, max:32, step:1}
simulation_distance = 10 #@param {type:"slider", min:4, max:32, step:1}
spawn_protection = 16 #@param {type:"integer"}
hardcore = False #@param {type:"boolean"}
generate_structures = True #@param {type:"boolean"}
enforce_secure_profile = True #@param {type:"boolean"}

import os

# 제공된 파일의 모든 항목을 기본값으로 설정
properties = {
    "accepts-transfers": "false",
    "allow-flight": str(allow_flight).lower(),
    "broadcast-console-to-ops": "true",
    "broadcast-rcon-to-ops": "true",
    "bug-report-link": "",
    "debug": "false",
    "difficulty": difficulty,
    "enable-code-of-conduct": "false",
    "enable-jmx-monitoring": "false",
    "enable-query": "false",
    "enable-rcon": "false",
    "enable-status": "true",
    "enforce-secure-profile": str(enforce_secure_profile).lower(),
    "enforce-whitelist": "false",
    "entity-broadcast-range-percentage": "100",
    "force-gamemode": "false",
    "function-permission-level": "2",
    "gamemode": gamemode,
    "generate-structures": str(generate_structures).lower(),
    "generator-settings": "{}",
    "hardcore": str(hardcore).lower(),
    "hide-online-players": "false",
    "initial-disabled-packs": "",
    "initial-enabled-packs": "vanilla",
    "level-name": "world",
    "level-seed": level_seed,
    "level-type": "minecraft\\:normal",
    "log-ips": "true",
    "max-chained-neighbor-updates": "1000000",
    "max-players": str(max_players),
    "max-tick-time": "60000",
    "max-world-size": "29999984",
    "motd": motd,
    "network-compression-threshold": "256",
    "online-mode": str(online_mode).lower(),
    "op-permission-level": "4",
    "pause-when-empty-seconds": "-1",
    "player-idle-timeout": "0",
    "prevent-proxy-connections": "false",
    "query.port": "25565",
    "rate-limit": "0",
    "rcon.password": "",
    "rcon.port": "25575",
    "region-file-compression": "deflate",
    "require-resource-pack": "false",
    "resource-pack": "",
    "resource-pack-id": "",
    "resource-pack-prompt": "",
    "resource-pack-sha1": "",
    "server-ip": "",
    "server-port": str(server_port),
    "simulation-distance": str(simulation_distance),
    "spawn-protection": str(spawn_protection),
    "status-heartbeat-interval": "0",
    "sync-chunk-writes": "true",
    "text-filtering-config": "",
    "text-filtering-version": "0",
    "use-native-transport": "true",
    "view-distance": str(view_distance),
    "white-list": str(white_list).lower(),
    # PVP 항목 누락 방지
    "pvp": str(pvp).lower()
}

# 관리 서버 관련 특수 설정 (제공된 값 유지)
management_props = {
    "management-server-allowed-origins": "",
    "management-server-enabled": "false",
    "management-server-host": "localhost",
    "management-server-port": "0",
    "management-server-secret": "qEhsR1BjY54BkCJgDy9IHt6IkzkeRS1MFBAJGTzB",
    "management-server-tls-enabled": "true",
    "management-server-tls-keystore": "",
    "management-server-tls-keystore-password": ""
}
properties.update(management_props)

# 파일 쓰기
target_path = '/content/server/server.properties'
with open(target_path, 'w') as f:
    f.write("#Minecraft server properties\n")
    import datetime
    f.write(f"#{datetime.datetime.now().strftime('%a %b %d %H:%M:%S KST %Y')}\n")
    for key, value in properties.items():
        f.write(f"{key}={value}\n")

print(f" 모든 설정이 {target_path}에 적용되었습니다.")

In [ ]:
# @title START SERVER { display-mode: "form" }

import subprocess
import time
import os

# --- 설정 (Form 사용) ---
# @markdown ### 서버 설정
jar_file_name = "paper.jar" # @param {type:"string"}
min_memory = "1G" # @param {type:"string"}
max_memory = "2G" # @param {type:"string"}
server_path = "/content/server" # @param {type:"string"}

# @markdown ---

# 1. 경로 확인 및 이동
if not os.path.exists(jar_file_name):
    try:
        os.chdir(server_path)
        print(f"알림: 작업 디렉토리를 {server_path}로 변경했습니다.")
    except FileNotFoundError:
        print(f"오류: {server_path} 경로를 찾을 수 없습니다.")

# 2. 실행 명령어 구성
server_command = [
    'java',
    f'-Xms{min_memory}',
    f'-Xmx{max_memory}',
    '-jar', jar_file_name,
    'nogui'
]

print(f"상태: {jar_file_name} 실행 중... (초기 파일 생성 및 로드)")

# 3. 서버 프로세스 실행
process = subprocess.Popen(
    server_command,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 4. 실시간 로그 모니터링 및 자동 종료
try:
    for line in iter(process.stdout.readline, ''):
        print(line.strip())

        # 서버 로딩 완료 감지
        if "Done (" in line:
            print("\n" + "-"*50)
            print("알림: 서버 로드가 완료되었습니다.")
            print("명령: 데이터를 저장하고 프로세스를 종료합니다...")
            print("-"*50 + "\n")

            time.sleep(2)
            process.stdin.write("stop\n")
            process.stdin.flush()
            break
except Exception as e:
    print(f"에러 발생: {e}")
    process.kill()

# 5. 종료 대기
process.wait(timeout=60)
print("\n결과: 서버가 안전하게 종료되었습니다.")

In [ ]:
#@title EXPORT SERVER FOLDER { run: "auto" }

#@markdown 다운로드할 압축 파일의 이름을 입력하세요.
zip_filename = "minecraft_server" #@param {type:"string"}

import os
from google.colab import files

# 확장자 처리 (사용자가 .zip을 직접 입력했을 경우 대비)
if not zip_filename.endswith(".zip"):
    full_filename = f"{zip_filename}.zip"
else:
    full_filename = zip_filename

# 압축 및 다운로드 진행
%cd /content
if os.path.exists('server'):
    print(f"Creating {full_filename}...")
    # !zip -r {zip_filename} server/ 명령어를 파이썬 변수와 함께 사용
    os.system(f"zip -r {full_filename} server/")

    print(f"\nDownloading {full_filename} to your computer...")
    files.download(full_filename)
else:
    print("Error: 'server' directory not found. Please run the previous cells first.")